# **DSK807: Applied Machine Learning**
## Exercise: Credit Card Default Prediction

In this exercise, you will work with a credit-card default dataset and compare several machine-learning approaches studied in the course.

**Main question:**  
> Which model would you use to predict credit-card default, and what evidence supports your choice?

## 1. Dataset

The dataset contains information about **30,000 credit-card customers**.

For each customer, information is available about:

- credit limit,
- demographic characteristics,
- payment status over the previous six months,
- monthly bill amounts,
- previous payment amounts.

The **target variable** indicates whether the customer defaulted on their credit-card payment in the following month.

Your first task is to understand the dataset before building any model.

More information about the dataset is available here: https://archive.ics.uci.edu/dataset/350/defaultofcreditcardclients

In [43]:
# Load the dataset here
import pandas as pd

from sklearn.metrics import precision_score, recall_score

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    AdaBoostClassifier,
    AdaBoostRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor
)
from xgboost import XGBClassifier, XGBRegressor


from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor


from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


from sklearn.metrics import roc_auc_score, average_precision_score
from time import time

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
)


import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
RT = 42

In [7]:
df = pd.read_csv('credit_card_clients.csv') # File is available on itslearning
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [2]:
df.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='str')

In [4]:
df['default payment next month'].value_counts()

default payment next month
0    23364
1     6636
Name: count, dtype: int64

In [36]:
y = df['default payment next month']
X = df.drop(columns=['default payment next month'])

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=RT,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15 / 0.85,
    random_state=RT,
    stratify=y_temp
)

In [ ]:
# whole model here

classification_models = {

    "Logistic Regression": make_pipeline(
        MinMaxScaler(),
        LogisticRegression(max_iter=5000, random_state=RT)
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        random_state=RT
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=1000,
        random_state=RT
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=RT
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=RT
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=RT,
        eval_metric="logloss"
    ),


    "Ridge (L2)": make_pipeline(StandardScaler(), LogisticRegression(penalty="l2", max_iter=5000)),
    "Lasso (L1)": make_pipeline(StandardScaler(), LogisticRegression(penalty="l1", solver="liblinear")),
    "KNN":        make_pipeline(StandardScaler(), KNeighborsClassifier()),
    'Dummy' : DummyClassifier(strategy="most_frequent")


}

In [39]:
# the initial test

classification_results = []
for name, model in classification_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    proba = model.predict_proba(X_val)[:, 1]


    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),

        "F1": f1_score(y_val, pred),
        "ROC AUC": roc_auc_score(y_val, proba),
        "PR AUC": average_precision_score(y_val, proba),   
    })

classification_results = (
    pd.DataFrame(classification_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

classification_results.round(3)

/Users/fangsiyu/dev-env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/fangsiyu/dev-env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/fangsiyu/dev-env/lib/python3.13/s

,Model,Accuracy,Precision,Recall,F1,ROC AUC,PR AUC
0,Gradient Boosting,0.829,0.727,0.362,0.483,0.783,0.568
1,XGBoost,0.827,0.718,0.358,0.478,0.783,0.568
2,Decision Tree,0.825,0.715,0.346,0.466,0.743,0.523
3,Random Forest,0.821,0.692,0.345,0.460,0.766,0.543
4,KNN,0.800,0.579,0.343,0.431,0.704,0.420
5,AdaBoost,0.820,0.727,0.301,0.425,0.774,0.560
6,Lasso (L1),0.809,0.725,0.222,0.340,0.732,0.524
7,Ridge (L2),0.809,0.722,0.222,0.340,0.732,0.524
8,Logistic Regression,0.808,0.716,0.220,0.337,0.731,0.523
9,Dummy,0.779,0.000,0.000,0.000,0.500,0.221


In [47]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RT)

spw = (y_train == 0).sum() / (y_train == 1).sum()

search_space = {
    "Logistic Regression": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(solver="liblinear", max_iter=5000, random_state=RT)),
        ]),
        {
            "clf__l1_ratio": [0, 1],
            "clf__C": [0.01, 0.1, 1, 10, 100],
            "clf__class_weight": [None, "balanced"],
        },
    ),

    "Decision Tree": (
        DecisionTreeClassifier(random_state=RT),
        {
            "max_depth": [3, 5, 7, 10, None],
            "min_samples_leaf": [1, 20, 50, 100],
            "class_weight": [None, "balanced"],
        },
    ),

    "Random Forest": (
        RandomForestClassifier(random_state=RT, n_jobs=1),
        {
            "n_estimators": [300],
            "max_depth": [None, 10, 20],
            "min_samples_leaf": [1, 5, 20],
            "max_features": ["sqrt", 0.3],
            "class_weight": [None, "balanced"],
        },
    ),

    "AdaBoost": (
        AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RT),
            random_state=RT,
        ),
        {
            "n_estimators": [100, 300, 500],
            "learning_rate": [0.05, 0.1, 0.5, 1.0],
            "estimator__max_depth": [1, 2],
        },
    ),

    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RT),
        {
            "n_estimators": [100, 300],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5],
            "subsample": [0.8, 1.0],
        },
    ),

    "XGBoost": (
        XGBClassifier(eval_metric="logloss", tree_method="hist", random_state=RT, n_jobs=1),
        {
            "n_estimators": [200, 400],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5],
            "subsample": [0.8],
            "colsample_bytree": [0.8, 1.0],
            "min_child_weight": [1, 5],
            "scale_pos_weight": [1, spw],
        },
    ),

    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier()),
        ]),
        {
            "clf__n_neighbors": [5, 15, 30, 50],
            "clf__weights": ["uniform", "distance"],
        },
    ),

    "Dummy": (
        DummyClassifier(strategy="most_frequent"),
        {},
    ),
}

rows = []
best_models = {}

for name, (model, grid) in search_space.items():
    t0 = time()

    search = GridSearchCV(
        model,
        grid,
        cv=cv,
        scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
        refit="pr_auc",
        n_jobs=-1,
    )
    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_
    i = search.best_index_
    cvr = search.cv_results_

    proba = search.predict_proba(X_val)[:, 1]
    pred = search.predict(X_val)   # 門檻 0.5

    rows.append({
        "Model fitted val": name,
        "CV PR AUC": cvr["mean_test_pr_auc"][i],
        "CV PR AUC std": cvr["std_test_pr_auc"][i],
        "CV ROC AUC": cvr["mean_test_roc_auc"][i],
        "Val PR AUC": average_precision_score(y_val, proba),
        "Val ROC AUC": roc_auc_score(y_val, proba),
        "Precision@0.5": precision_score(y_val, pred, zero_division=0),
        "Recall@0.5": recall_score(y_val, pred),
        "F1@0.5": f1_score(y_val, pred),
        # "Best params": search.best_params_,
        "Time (s)": round(time() - t0),
    })
    print(f"{name}done, cost {rows[-1]['Time (s)']} sec")

grid_results = (
    pd.DataFrame(rows)
    .sort_values("CV PR AUC", ascending=False)
    .reset_index(drop=True)
)
grid_results.round(3)

Logistic Regressiondone, cost 4 sec
Decision Treedone, cost 3 sec
Random Forestdone, cost 149 sec
AdaBoostdone, cost 110 sec
Gradient Boostingdone, cost 86 sec
XGBoostdone, cost 12 sec
KNNdone, cost 1 sec
Dummydone, cost 0 sec


,Model fitted val,CV PR AUC,CV PR AUC std,CV ROC AUC,Test PR AUC,Test ROC AUC,Precision@0.5,Recall@0.5,F1@0.5,Time (s)
0,XGBoost,0.561,0.020,0.783,0.569,0.782,0.493,0.606,0.543,12
1,Gradient Boosting,0.559,0.018,0.783,0.578,0.785,0.724,0.370,0.490,86
2,Random Forest,0.558,0.019,0.780,0.573,0.783,0.738,0.354,0.478,149
3,AdaBoost,0.544,0.017,0.776,0.562,0.772,0.733,0.337,0.461,110
4,Decision Tree,0.522,0.014,0.761,0.546,0.769,0.700,0.366,0.481,3
5,KNN,0.507,0.013,0.745,0.521,0.753,0.685,0.281,0.399,1
6,Logistic Regression,0.501,0.015,0.721,0.526,0.731,0.743,0.200,0.315,4
7,Dummy,0.221,0.000,0.500,0.221,0.500,0.000,0.000,0.000,0


In [49]:
# evaluate via test

classification_results = []
for name, model in best_models.items():
    # model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    classification_results.append({
        "Model fi": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0), # 分母也是 0，數學上沒有定義
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC AUC": roc_auc_score(y_test, proba),
        "PR AUC": average_precision_score(y_test, proba),
    })

classification_results = (
    pd.DataFrame(classification_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

classification_results.round(3)

,Model fi,Accuracy,Precision,Recall,F1,ROC AUC,PR AUC
0,XGBoost,0.768,0.480,0.618,0.540,0.780,0.562
1,Decision Tree,0.820,0.675,0.362,0.471,0.765,0.538
2,Gradient Boosting,0.820,0.680,0.354,0.465,0.780,0.562
3,Random Forest,0.820,0.684,0.350,0.463,0.781,0.562
4,AdaBoost,0.819,0.698,0.321,0.439,0.776,0.551
5,KNN,0.808,0.654,0.279,0.392,0.747,0.521
6,Logistic Regression,0.808,0.716,0.218,0.334,0.706,0.498
7,Dummy,0.779,0.000,0.000,0.000,0.500,0.221


In [55]:
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.metrics import (
    make_scorer, fbeta_score, precision_score, recall_score,
    f1_score, accuracy_score, confusion_matrix,
    roc_auc_score, average_precision_score,
)


def recall_at_min_precision(y_true, y_pred, min_precision):
    if precision_score(y_true, y_pred, zero_division=0) < min_precision:
        return 0.0
    return recall_score(y_true, y_pred)

precision_floor_scorer = make_scorer(recall_at_min_precision, min_precision=0.5)

# f2_scorer = make_scorer(fbeta_score, beta=2)

tuned_models = {}
for name, model in best_models.items():
    if name == "Dummy":
        continue
    tuned = TunedThresholdClassifierCV(
        estimator=model,
        scoring=precision_floor_scorer,
        cv="prefit",
        refit=False,
    )
    tuned.fit(X_val, y_val)
    tuned_models[name] = tuned
    print(f"{name}: threshold={tuned.best_threshold_:.3f}, val F2={tuned.best_score_:.3f}")



# tuned_models['Dummy'] = best_models["Dummy"]
classification_threshold_results = []
for name, tuned in tuned_models.items():
    pred = tuned.predict(X_test)
    proba = tuned.predict_proba(X_test)[:, 1]

    classification_threshold_results.append({
        "Model fit test": name,
        "Threshold": tuned.best_threshold_,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "F2": fbeta_score(y_test, pred, beta=2),
        "ROC AUC": roc_auc_score(y_test, proba),
        "PR AUC": average_precision_score(y_test, proba),
    })

classification_threshold_results_df = (
    pd.DataFrame(classification_threshold_results)
    .sort_values("F2", ascending=False)
    .reset_index(drop=True)
)

classification_threshold_results_df.round(3)

Logistic Regression: threshold=0.249, val F2=0.559
Decision Tree: threshold=0.278, val F2=0.560
Random Forest: threshold=0.239, val F2=0.597
AdaBoost: threshold=0.400, val F2=0.579
Gradient Boosting: threshold=0.237, val F2=0.599
XGBoost: threshold=0.514, val F2=0.600
KNN: threshold=0.246, val F2=0.540


,Model fit test,Threshold,Accuracy,Precision,Recall,F1,F2,ROC AUC,PR AUC
0,Gradient Boosting,0.237,0.780,0.502,0.603,0.548,0.580,0.780,0.562
1,Random Forest,0.239,0.781,0.504,0.594,0.545,0.573,0.781,0.562
2,XGBoost,0.514,0.772,0.488,0.598,0.537,0.572,0.780,0.562
3,AdaBoost,0.400,0.776,0.494,0.564,0.527,0.548,0.776,0.551
4,Decision Tree,0.278,0.777,0.496,0.555,0.523,0.542,0.765,0.538
5,Logistic Regression,0.249,0.764,0.471,0.547,0.506,0.530,0.706,0.498
6,KNN,0.246,0.770,0.482,0.527,0.503,0.517,0.747,0.521


In [ ]:
# for x in best_models:：只拿到 key，所以 x 依序是 "XGBoost"、"KNN" 這些字串。這就是你上一版出錯的原因：把字串當成模型傳進去。
# for x in best_models.values():：只拿到 value，也就是模型本身，但這樣就不知道是哪個模型。
# for x in best_models.items():：每次拿到一組 (key, value)，例如 ("XGBoost", 訓練好的 XGBClassifier)

這是基於 f2_scorer = make_scorer(fbeta_score, beta=2) 跑出的

$$F_2 = \frac{5 \cdot P \cdot R}{4P + R}$$

Logistic Regression: threshold=0.112, val F2=0.590

Decision Tree: threshold=0.086, val F2=0.627

Random Forest: threshold=0.141, val F2=0.633

AdaBoost: threshold=0.332, val F2=0.631

Gradient Boosting: threshold=0.129, val F2=0.635

XGBoost: threshold=0.296, val F2=0.636

KNN: threshold=0.097, val F2=0.612



| Model fit test      | Threshold | Accuracy | Precision | Recall | F1    | F2    | ROC AUC | PR AUC |
|---------------------|----------:|---------:|----------:|-------:|------:|------:|--------:|-------:|
| Random Forest       | 0.141     | 0.610    | 0.340     | 0.809  | 0.479 | 0.634 | 0.781   | 0.562  |
| Gradient Boosting   | 0.129     | 0.577    | 0.323     | 0.834  | 0.466 | 0.634 | 0.780   | 0.562  |
| XGBoost             | 0.296     | 0.536    | 0.305     | 0.863  | 0.451 | 0.632 | 0.780   | 0.562  |
| AdaBoost            | 0.332     | 0.531    | 0.303     | 0.860  | 0.448 | 0.629 | 0.776   | 0.551  |
| Decision Tree       | 0.086     | 0.468    | 0.281     | 0.902  | 0.428 | 0.625 | 0.765   | 0.538  |
| KNN                 | 0.097     | 0.447    | 0.272     | 0.893  | 0.417 | 0.613 | 0.747   | 0.521  |
| Logistic Regression | 0.112     | 0.351    | 0.240     | 0.891  | 0.378 | 0.577 | 0.706   | 0.498  |
